# OBJECTIVE: Road Network Standardization - Fire Station Sector

This notebook performs a comprehensive cleaning and normalization of all 
road names within the Fire Station's intervention zone. 

KEY GOALS:
* Data Cleaning: Standardize naming conventions to remove inconsistencies 
  (e.g., abbreviations, typos, and varying formats).
* Deduplication: Resolve overlaps to establish a 'Single Source of Truth' 
  for the road network.
* Unique Master List: Generate the definitive list of all unique routes 
  within the sector to optimize emergency response routing and GIS accuracy.

In [ ]:
import pandas as pd

df = pd.read_csv("Merge_address.csv", sep=',')
df.shape
df.head()


/tmp/ipykernel_81377/4226230325.py:3: DtypeWarning: Columns (0: hrmn, 1: com, 2: gps, 3: lat, 4: long, 5: dep) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("caracteristiques-2005-2024.csv", sep=',')


,Unnamed: 0,an,mois,jour,hrmn,lum,agg,int,atm,col,com,adr,gps,lat,long,dep,Source
0,200500000001,5,1,12,1900,3,2,1,1.0,3.0,11.0,CD41B,M,5051500.0,294400.0,590,2005
1,200500000002,5,1,21,1600,1,2,1,1.0,1.0,51.0,rue de Lille,M,5053700.0,280200.0,590,2005
2,200500000003,5,1,21,1845,3,1,1,2.0,1.0,51.0,NaN,M,5054600.0,280000.0,590,2005
3,200500000004,5,1,4,1615,1,1,1,1.0,5.0,82.0,NaN,M,5098700.0,240800.0,590,2005
4,200500000005,5,1,10,1945,3,1,1,3.0,6.0,478.0,NaN,M,5096400.0,247500.0,590,2005


Filter by municipality code to retain only the records belonging to the fire station's specific sector.

In [10]:
secteur = {    
    "Bassens": 33032,
    "Sainte-Eulalie": 33397,
    "Carbon-Blanc":33096,
    "Yvrac":33554,
    "Ambarès-et-Lagrave":33003,
    "Lormont": 33249
}

secteur_com_list = [str(code) for code in secteur.values()]

print(secteur_com_list)
df_secteur = df[df['com'].isin(secteur_com_list)]
print(df_secteur.shape)
df_secteur.head(5)

['33032', '33397', '33096', '33554', '33003', '33249']
(502, 17)


,Unnamed: 0,an,mois,jour,hrmn,lum,agg,int,atm,col,com,adr,gps,lat,long,dep,Source
959381,201900000913,2019,9,20,07:30,1,1,1,1.0,3.0,33249,ROCADE A 630,NaN,"44,8846500","-0,5157800",33,2019
959747,201900001279,2019,5,16,15:20,1,1,1,1.0,2.0,33096,AUTOROUTE A 10,NaN,"44,8935300","-0,4994600",33,2019
959748,201900001280,2019,5,14,11:10,1,1,1,1.0,2.0,33096,AUTOROUTE A 10,NaN,"44,9007100","-0,4943200",33,2019
959756,201900001288,2019,4,30,18:20,1,1,1,1.0,5.0,33397,AUTOROUTE A 10,NaN,"44,9082600","-0,4904000",33,2019
960402,201900001934,2019,11,28,10:30,1,1,1,1.0,3.0,33096,AUTOROUTE A 10,NaN,"44,9094000","-0,4894200",33,2019


Clean road name avec export

In [ ]:
import unidecode
import re




def clean_road_data(clean_adr):
    # Initial cleaning
    clean_adr = clean_adr.astype(str).str.lower().str.strip()
    clean_adr = clean_adr.str.encode('latin1').str.decode('utf-8', errors='ignore').fillna("")

    # Remove street numbers
    clean_adr = clean_adr.str.replace(r'^\d+\s*(bis|ter|quater)?\s*', '', regex=True)

    # Remove narrative noise and technical details
    clean_adr = clean_adr.apply(lambda x: re.split(r' - |\(| sur | venant | vta | vt ', x)[0].strip())

    # Normalize characters
    clean_adr = clean_adr.apply(lambda x: unidecode.unidecode(x))

    # Replace common abbreviations and errors
    replacements = {
        r'libration': 'libération',
        r'rpublique': 'république',
        r'prsident': 'président',
        r'marchal': 'maréchal',
        r'flix': 'félix',
        r'franois': 'françois',
        r'clmenceau': 'clémenceau',
        r'ambars|d’ambares': "d'ambarès",
        r'gravires': 'gravières',
        r'\bave\b': 'avenue',
        r'\brte\b': 'route',
        r'\brn\b': 'route nationale',
        r'\bav\b': 'avenue',
        r'\bst\b': 'saint',
        r'\b(rn|n|route nationale)\s*230\b': 'route nationale 230',
        r'\b(a|autoroute)\s*630\b': 'autoroute a630',
        r'\b(autoroute\s+a\s*10|a\s*10)\b': 'autoroute a10',
        "autoroute autoroute": "autoroute",
        r"d'd'": "d'",
        r"accs": "acces",
        r"cte de": "cote de",
        r"lon blum": "leon blum",
        r"andr ricard": "andre ricard",
        r" -rd \d+": "",
        r"place de leglise": "place de l'eglise",
        "ctre cial": "centre commercial",
        "ccial": "centre commercial",
    }

    # Simplify major axes
    clean_adr = clean_adr.str.replace(r'.*(route nationale 230|autoroute a630|autoroute a10).*', r'\1', regex=True)

    for pattern, replacement in replacements.items():
        clean_adr = clean_adr.str.replace(pattern, replacement, regex=True)

    # Remove empty lines and process intersections
    clean_adr = clean_adr[(clean_adr != "") & (clean_adr != "sur parking")]
    clean_adr = clean_adr.apply(lambda x: x.split(' / ')[0].strip())

    return clean_adr

# Application
clean_adr = df_secteur["adr"].astype(str).str.lower().str.strip()
clean_adr = clean_road_data(clean_adr)

# Export de la liste unique
clean_adr.drop_duplicates().sort_values().to_csv('liste_communes_nettoyee.csv', index=False)